In [ ]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_key", "", "Merge Key")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_key = dbutils.widgets.get("merge_key")
load_type = dbutils.widgets.get("load_type")

In [ ]:
bronze_df = spark.read.table(f"adb_healthy.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [ ]:
from pyspark.sql import functions as F

updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

In [ ]:
from delta.tables import DeltaTable

target_table = f"adb_healthy.silver.{table_name}"
load_type_normalized = (load_type or "").strip().upper()
target_exists = spark.catalog.tableExists(target_table)

if load_type_normalized in ("", "NULL", "NONE"):
    # Full load: overwrite the target table
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type_normalized == "APPEND":
    # Append new records to the target table (creates it if missing)
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type_normalized == "MERGE":
    # Upsert: update existing records and insert new ones based on merge key(s)
    merge_keys = [k.strip() for k in (merge_key or "").split(",") if k.strip()]
    if not merge_keys:
        raise ValueError("merge_key must be provided (comma-separated for composite keys) when load_type = 'MERGE'")

    missing_keys = [k for k in merge_keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(f"Merge key(s) {missing_keys} not found in source columns")

    if not target_exists:
        # First run: nothing to merge into, create the table
        updated_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        # Deduplicate source on merge keys to avoid multiple-match merge errors
        source_df = updated_df.dropDuplicates(merge_keys)

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in merge_keys])
        # Preserve the original insert_timestamp on updates
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}

        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute()
        )

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected NULL/empty, 'APPEND', or 'MERGE'.")

In [ ]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))